# StudyMate · Part 4 of 8: Answer Cards and Guard

**Assignment part:** Core Functionality (RAG)  
**Group 7:** Sadia Saeed · Eva Abouabouharb · Neida Guzman

**Qwen2.5-3B-Instruct** writes the 4-part answer card (simple explanation, real-life example, memory trick, flashcard) from the top 2 lessons only. The **off-topic guard** blocks a question only if both the hybrid and reranker scores are low.

> 📖 **This notebook is for reading.** It is one section of the full notebook, with the real outputs kept. Each part uses results from the parts before it, so to **run** StudyMate, open [`FN_StudyMate_Group7_ITAI2377.ipynb`](FN_StudyMate_Group7_ITAI2377.ipynb) and use **Runtime → Run all**.

## 5. Answer Cards with Qwen (RAG: Retrieve, Then Generate)

**Purpose:** Turn the best lesson into a beginner-friendly answer card.

**How it works:**
1. `search()` finds the top 3 lessons for the question.
2. The #1 lesson's text is given to **Qwen2.5-1.5B-Instruct** as the only facts it may use.
3. Qwen writes a card with: simple explanation, real-life example, memory trick, and flashcard.
4. The other 2 lessons are shown as related lessons.

**Honesty rules:** Qwen must use only the lesson's facts, and must say so when the lesson doesn't answer the question. Greedy decoding (no random sampling) keeps answers steady and reduces invented details.

**Output:** Answer cards for 3 ML questions, plus 1 off-topic question to test the honesty rule.


In [38]:
# CODE CELL 5 — ANSWER CARDS WITH QWEN (RAG)
import re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import Markdown, display

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then re-run Cells 1–4."

llm_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=torch.float16, device_map="auto")

# Off-topic guard: "Sorry" only if BOTH scores are low (thresholds set from the dev questions)
HYBRID_MIN = 0.18
RERANK_MIN = -9.0
SORRY = "Sorry, StudyMate doesn't have a lesson on that yet. Try asking about a machine learning topic!"
SECTIONS = ["Simple explanation", "Real-life example", "Memory trick", "Flashcard"]

SYSTEM = (
    "You are StudyMate, a friendly machine learning tutor for beginners. "
    "Use ONLY the facts in the lessons provided. Use mainly the MAIN lesson; use the RELATED lesson "
    "only if the question also asks about it. Do not add facts or numbers that are not in the lessons. "
    "Write in simple English for an 8th-grade student, but keep correct technical terms.\n\n"
    "Answer in exactly this format, with every section included:\n"
    "**Simple explanation:** 2-3 sentences.\n"
    "**Real-life example:** 1-2 sentences from healthcare, retail, or everyday life.\n"
    "**Memory trick:** one short, catchy line about this topic.\n"
    "**Flashcard:** Q: one short question A: one short answer"
)

def is_off_topic(question, top3=None):
    """Returns (blocked?, best hybrid score, best reranker score)."""
    top3 = search(question) if top3 is None else top3
    h = float(hybrid_scores(question).max())
    r = float(top3["rerank_score"].max())
    return (h < HYBRID_MIN and r < RERANK_MIN), h, r

def lesson_context(row):
    parts = [f"Title: {row['title']}",
             f"Simple explanation: {row['simple_explanation']} {row['extra_simple']}".strip(),
             f"Details: {row['detailed_explanation']}"]
    if row["real_world_example"]:
        parts.append(f"Example: {row['real_world_example']}")
    return "\n".join(parts)

def tidy(card):
    """Put each section on its own line and the flashcard answer on a new line."""
    card = re.sub(r"\s*(\*\*(?:Real-life example|Memory trick|Flashcard):\*\*)", r"\n\n\1", card)
    card = re.sub(r"\s+A:", "  \nA:", card)
    return card.strip()

def answer(question):
    top3 = search(question)                                   # retrieve (Cell 4)
    blocked, h, r = is_off_topic(question, top3)              # off-topic guard
    if blocked:
        return {"question": question, "lesson": "—", "card": SORRY, "related": [],
                "hybrid_max": h, "rerank_max": r, "missing": []}

    main, related = top3.iloc[0], top3.iloc[1]
    user_msg = (f"Student question: {question}\n\nMAIN lesson:\n{lesson_context(main)}"
                f"\n\nRELATED lesson:\n{lesson_context(related)}")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():                                     # generate
        out = llm.generate(**inputs, max_new_tokens=350, do_sample=False, repetition_penalty=1.1)
    card = tidy(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    missing = [s for s in SECTIONS if f"**{s}:**" not in card]
    return {"question": question, "lesson": main["title"], "card": card,
            "related": top3["title"].tolist()[1:], "hybrid_max": h, "rerank_max": r, "missing": missing}

def show_answer(r):
    warn = f"\n\n⚠️ **Missing:** {', '.join(r['missing'])}" if r["missing"] else ""
    rel = f"\n\n**🔗 Related lessons:** {', '.join(r['related'])}" if r["related"] else ""
    display(Markdown(f"### ❓ {r['question']}\n**📘 Lesson used:** {r['lesson']}\n\n{r['card']}{warn}{rel}"))

# Demo: 3 ML questions + 1 off-topic question
for q in ["What is the difference between precision and recall?",
          "How does gradient descent help a model learn?",
          "How do I turn a column of city names into numbers?",
          "What is the capital of France?"]:
    start = time.time()
    r = answer(q)
    show_answer(r)
    print(f"Hybrid max: {r['hybrid_max']:.3f} | Reranker max: {r['rerank_max']:.2f} | "
          f"⏱ {time.time() - start:.1f} seconds\n")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### ❓ What is the difference between precision and recall?
**📘 Lesson used:** F1 Score

**Simple explanation:** Precision is how many true positive results there are compared to all positive results given by the test, while recall is how many of the actual positives were found by the test.

**Real-life example:** In a medical test for a disease, high precision means few patients who don't have the disease are incorrectly diagnosed as having it, whereas high recall means most people who actually have the disease are correctly identified.

**Memory trick:** "Precision checks if you caught the right bad guys, recall counts all the bad guys."

**Flashcard:** Q: What does precision measure?  
A: How many true positive results there are compared to all positive results given by the test.

**🔗 Related lessons:** Recall, Precision

Hybrid max: 0.473 | Reranker max: 3.85 | ⏱ 17.2 seconds



### ❓ How does gradient descent help a model learn?
**📘 Lesson used:** Gradient Descent

**Simple explanation:** Gradient Descent helps a model learn by making tiny changes to its settings over many tries, just like a ball rolls down a hill to reach the bottom. It keeps trying different paths until it finds the best way to make predictions without making too many mistakes.

**Real-life example:** In healthcare, doctors might tweak a medical diagnosis tool using Gradient Descent to improve accuracy. They start with initial guesses and then test them against patient data, adjusting the tool slightly each time to see if it gets better at predicting outcomes.

**Memory trick:** "Gradient Descent rolls down hills to win!"

**Flashcard:** Q: What does Gradient Descent do?  
A: It makes tiny changes to model settings to reduce errors.

**🔗 Related lessons:** Loss and Optimizers, Backpropagation

Hybrid max: 0.613 | Reranker max: 7.11 | ⏱ 19.2 seconds



### ❓ How do I turn a column of city names into numbers?
**📘 Lesson used:** Embedding

**Simple explanation:** To turn a column of city names into numbers, we can create a special dictionary where each unique city name gets a different number. This way, instead of having text, we have numbers that represent those cities.

**Real-life example:** In a retail store, they might use city names to track customer locations. By turning these city names into numbers, they can easily sort or group customers based on their location without needing to understand the actual city names.

**Memory trick:** "City Names Turn Into Numbers, Just Like Red Blue Green!"

**Flashcard:**
Q: How can you convert city names into numbers?  
A: Create a dictionary where each city name maps to a unique number.

**🔗 Related lessons:** One-Hot Encoding, Text Features

Hybrid max: 0.213 | Reranker max: -10.47 | ⏱ 14.8 seconds



### ❓ What is the capital of France?
**📘 Lesson used:** —

Sorry, StudyMate doesn't have a lesson on that yet. Try asking about a machine learning topic!

Hybrid max: 0.079 | Reranker max: -11.09 | ⏱ 0.1 seconds

